# 📈 Sales & Demand Forecasting: A Business Intelligence Approach

**Project by**: Charan  
**Role**: Data Scientist (ML Intern)  
**Objective**: Predicting future demand to optimize inventory and business planning.

---

## 1. Introduction & Business Context

Welcome! In this project, I am taking on the role of a **Data Scientist (ML Intern)** working for a retail business. 

### Why is this important?
Sales forecasting is not just about numbers; it's about **Business Strategy**. Companies use these models to:
- **Plan Inventory**: Avoid overstocking (which ties up cash) or understocking (which loses customers).
- **Staffing**: Predict when the warehouse will be busiest.
- **Cash Flow**: Prepare for periods of high outflow for stock buildup.

Our goal today is to take raw transactional data and turn it into a clear, 8-week sales roadmap.

## 2. Setting Up the Workspace

Before we dive into the data, we need to set up our environment. We are using a **Modular Approach**, which means we've separated our heavy logic into specialized files in the `src/` folder. This makes our notebook clean and our code reusable for web apps like Streamlit or Flask later.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Adding the project root to sys.path so we can import our custom modules
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Importing our modular logic
from src.data_loader import load_raw_data, load_custom_file
from src.preprocessing import clean_data, create_features
from src.visualization import plot_monthly_trend, plot_forecast_vs_actual, plot_future_forecast
from src.forecasting import train_model, evaluate_model, save_model, predict_recursive
from src.utils import ensure_directories, CLEAN_DATA_PATH

ensure_directories()
print("✅ Environment Ready!")

## 3. Data Ingestion & Custom Loader

We typically work with the **Online Retail** dataset. However, we've also implemented a `load_custom_file` function that allows us to audit other datasets (CSV or Excel) easily.

### Loading the Standard Dataset:

In [ ]:
raw_df = load_raw_data()
print(f"Standard Data Loaded: {raw_df.shape[0]:,} rows across {raw_df.shape[1]} columns.")
raw_df.head()

### Auditing a Custom Dataset:
If you have another dataset, you can load it here. (Uncomment the lines below to use).

In [ ]:
# custom_file_path = 'data/your_other_dataset.csv'
# if Path(custom_file_path).exists():
#     with open(custom_file_path, 'rb') as f:
#         custom_df = load_custom_file(f)
#         print(f"Custom Data Loaded: {custom_df.shape}")

## 4. Data Cleaning: Preparing the Foundation

Real-world data is messy. In this step, we are:
1. **Removing Cancellations**: Orders starting with 'C' are returns, which would skew our demand predictions.
2. **Filtering Negative Values**: We only want actual sales (Quantity > 0 and UnitPrice > 0).
3. **Date Conversion**: Turning text dates into Python datetime objects so we can extract time patterns.

In [ ]:
df = clean_data(raw_df)
df.to_csv(CLEAN_DATA_PATH, index=False)
print(f"🧹 Cleaning Complete! We are left with {len(df):,} high-quality records.")

## 5. Exploratory Data Analysis (EDA)

Let's visualize our history. We'll look at the **Monthly Sales Trend** to see if the business is growing and identify any obvious seasonality.

In [ ]:
monthly_plot = plot_monthly_trend(df)

# Identifying the strongest markets
top_countries = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(3)
print("🌍 Top 3 Revenue Markets:")
for country, revenue in top_countries.items():
    print(f"- {country}: ${revenue:,.2f}")

## 6. Feature Engineering: Teaching the Model to Remember

Machine Learning models don't naturally understand that 'today' follows 'yesterday'. We teach them this by creating **Lag Features**:
- **Sales_Lag_1**: What were the sales last week?
- **Sales_Rolling_4**: What was the average of the last month?

We aggregate the data to a **Weekly frequency** because daily sales are too noisy for long-term inventory planning.

In [ ]:
weekly_data = create_features(df, frequency='W')
print("⚙️ Features Engineered. Each row now contains history that helps the model learn patterns.")
weekly_data.head()

## 7. Model Training & Validation

We use a **Random Forest** approach. It's powerful, robust to outliers, and great for capturing the non-linear surges we see in holiday shopping seasons.

We train on everything *except* the last 8 weeks, which we use to 'test' our model's accuracy on data it has never seen.

In [ ]:
test_size = 8
train = weekly_data.iloc[:-test_size]
test = weekly_data.iloc[-test_size:]

X_train = train.drop(['Date', 'Sales'], axis=1)
y_train = train['Sales']
X_test = test.drop(['Date', 'Sales'], axis=1)
y_test = test['Sales']

model = train_model(X_train, y_train, model_type='RF')
y_pred = model.predict(X_test)

metrics = evaluate_model(y_test, y_pred)
print(f"📊 Validation R2 Score: {metrics['R2']:.2f}")
plot_forecast_vs_actual(train, test, y_pred)

## 8. The Roadmap: 8-Week Future Forecast

Now the moment of truth! We take the very last data point and use our model to recursively predict the next 8 weeks. This is what a Store Manager would use to plan their next quarter.

In [ ]:
future_dates = pd.date_range(start=weekly_data['Date'].iloc[-1] + pd.Timedelta(weeks=1), periods=8, freq='W')
last_row_features = X_test.iloc[-1:].copy()
future_forecast = predict_recursive(model, last_row_features, steps=8)

plot_future_forecast(weekly_data, future_dates, future_forecast)
save_model(model, 'weekly_sales_model.pkl')
print("✨ 8-Week Business Forecast Generated!")

## 9. Final Insights & Recommendations

### 💡 Key Findings
1. **Holiday Surge**: There is a clear, statistically significant surge in Q4. We recommend a **30% increase** in stock levels starting October.
2. **Market Strategy**: The UK is the core market, but the model identifies rising trends in Germany and France—good targets for holiday marketing.
3. **Model Stability**: Weekly forecasting is **much more accurate** for this business than daily forecasting, which is too volatile for warehouse planning.

### 🚀 Next Steps
- **Web Dashboard**: Deploy this model via Streamlit so managers can check forecasts in real-time.
- **Feature Growth**: Add 'Holiday Flags' (e.g., Black Friday) to the feature set to further improve surge accuracy.

---